In [1]:
# ---- LOAD THE WEEK 6 COMPARISON DATA ----
import os
import json
import glob

BASE = os.path.expanduser('~/easyf1')
results_dir = os.path.join(BASE, 'results')

# Find the full-15 comparison file
files = sorted(glob.glob(os.path.join(results_dir, 'comparison_full15_*.json')))
# Exclude the _meta files
data_files = [f for f in files if not f.endswith('_meta.json')]

print("Found comparison files:")
for f in data_files:
    print(f"  {os.path.basename(f)}")

# Load the most recent one
latest = data_files[-1]
with open(latest, 'r') as f:
    scenarios_data = json.load(f)

print(f"\nLoaded: {os.path.basename(latest)}")
print(f"Scenarios: {len(scenarios_data)}")
print(f"\nEach scenario has these keys: {list(scenarios_data[0].keys())}")
print(f"\nFirst scenario preview:")
print(f"  Scenario: {scenarios_data[0]['scenario'][:100]}...")
print(f"  Context chunks: {len(scenarios_data[0]['context'])}")
print(f"  Has Gemini answer: {'gemini' in scenarios_data[0]}")
print(f"  Has Nemotron answer: {'nemotron' in scenarios_data[0]}")

Found comparison files:
  comparison_full15_20260907_184605.json

Loaded: comparison_full15_20260907_184605.json
Scenarios: 15

Each scenario has these keys: ['scenario', 'context', 'gemini', 'nemotron', 'scenario_num']

First scenario preview:
  Scenario: It's lap 32 of 52 at the 2023 British Grand Prix (Silverstone). Lewis Hamilton is P4 on 30-lap-old h...
  Context chunks: 6
  Has Gemini answer: True
  Has Nemotron answer: True


In [2]:

# 5 additional scenarios in addition to previous 15


scenarios_batch3 = [
    # 16. High-degradation street circuit, undercut vs track position
    "It's lap 18 of 50 at the 2022 Azerbaijan Grand Prix (Baku). A driver is "
    "P5 on 16-lap-old soft tyres, struggling with degradation on the long "
    "straights. The car behind is closing. Should they pit now for hards, or "
    "risk staying out to hold track position?",

    # 17. Late-race two-stopper catching a one-stopper (strategy divergence)
    "It's lap 40 of 53 at the 2023 Spanish Grand Prix. A driver on a two-stop "
    "strategy is P6 on fresh softs, 8 seconds behind a one-stopper in P5 who is "
    "on 25-lap-old hards. Is there enough time and tyre advantage to catch and "
    "pass before the finish?",

    # 18. Weather gamble — dry to wet (opposite of the earlier wet scenario)
    "It's lap 20 of 44 at a race where rain has just started falling lightly. "
    "A driver is P3 on slick tyres. Some rivals are staying out. Should they "
    "pit for intermediates now or wait to see if the rain intensifies?",

    # 19. Safety car right before a planned stop (timing windfall)
    "It's lap 26 of 58 at the 2023 Qatar Grand Prix. A driver in P4 was "
    "planning to pit in 2-3 laps anyway when a Safety Car is suddenly deployed. "
    "Should they dive into the pits immediately to take the cheap stop?",

    # 20. Long opening stint decision at a low-deg circuit
    "It's lap 15 of 44 at the 2021 Italian Grand Prix (Monza), a low-degradation "
    "circuit. A driver is P4 on medium tyres that still feel strong. Should they "
    "extend this stint well past the normal window to gain track position through "
    "strategic offset?",
]

print(f"Batch 3: {len(scenarios_batch3)} new scenarios (16-20)")
for i, s in enumerate(scenarios_batch3, 16):
    print(f"\n{i}. {s[:75]}...")

Batch 3: 5 new scenarios (16-20)

16. It's lap 18 of 50 at the 2022 Azerbaijan Grand Prix (Baku). A driver is P5 ...

17. It's lap 40 of 53 at the 2023 Spanish Grand Prix. A driver on a two-stop st...

18. It's lap 20 of 44 at a race where rain has just started falling lightly. A ...

19. It's lap 26 of 58 at the 2023 Qatar Grand Prix. A driver in P4 was planning...

20. It's lap 15 of 44 at the 2021 Italian Grand Prix (Monza), a low-degradation...


In [3]:

# Import engine functions + run batch 3

import sys
import os

BASE = os.path.expanduser('~/easyf1')

# Add the app folder to the path so we can import engine.py
sys.path.insert(0, os.path.join(BASE, 'app'))
import engine

print("Engine imported ✅")

# Run the 5 new scenarios through both models
import time

batch3_results = []
for idx, scenario in enumerate(scenarios_batch3):
    scenario_num = idx + 16
    print(f"\n{'#'*55}")
    print(f"# SCENARIO {scenario_num} of 20")
    print(f"{'#'*55}")
    print(f"{scenario[:90]}...")
    
    # engine.compare returns dict with scenario, context, gemini, nemotron
    result = engine.compare(scenario)
    result['scenario_num'] = scenario_num
    batch3_results.append(result)
    
    # Quick preview
    print(f"\n  Gemini: {result['gemini'][:60]}...")
    print(f"  Nemotron: {result['nemotron'][:60]}...")
    
    if idx < len(scenarios_batch3) - 1:
        print("\n[pausing 6s...]")
        time.sleep(6)

print(f"\n\n{'='*55}")
print(f"BATCH 3 COMPLETE: {len(batch3_results)} scenarios ✅")
print(f"{'='*55}")

Engine imported ✅

#######################################################
# SCENARIO 16 of 20
#######################################################
It's lap 18 of 50 at the 2022 Azerbaijan Grand Prix (Baku). A driver is P5 on 16-lap-old s...


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



  Gemini: VERDICT: Pit now for hard tyres to mitigate degradation and ...
  Nemotron: VERDICT: Pit now for hard tyres to avoid losing track positi...

[pausing 6s...]

#######################################################
# SCENARIO 17 of 20
#######################################################
It's lap 40 of 53 at the 2023 Spanish Grand Prix. A driver on a two-stop strategy is P6 on...

  Gemini: VERDICT: Attempt the overtake, as the fresh soft tyres offer...
  Nemotron: VERDICT: Yes, push to catch and pass P5 before the finish.
C...

[pausing 6s...]

#######################################################
# SCENARIO 18 of 20
#######################################################
It's lap 20 of 44 at a race where rain has just started falling lightly. A driver is P3 on...

  Gemini: VERDICT: Pit for intermediate tyres now.
CONFIDENCE: Medium
...
  Nemotron: VERDICT: Pit for intermediates now rather than waiting to se...

[pausing 6s...]

#########################################

In [4]:

# Merging all 20 scenarios + saving as a evaluation set
import json
import pickle
from datetime import datetime

# scenarios_data holds the original 15 (loaded earlier)
# batch3_results holds the new 5
full_20 = scenarios_data + batch3_results
full_20.sort(key=lambda x: x['scenario_num'])

# Verify we have a clean 1-20
nums = [r['scenario_num'] for r in full_20]
print(f"Total scenarios: {len(full_20)}")
print(f"Scenario numbers: {nums}")

# Save as the official evaluation dataset
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
eval_dir = os.path.join(BASE, 'results')

json_path = os.path.join(eval_dir, f'eval_dataset_20_{timestamp}.json')
with open(json_path, 'w') as f:
    json.dump(full_20, f, indent=2)

pkl_path = os.path.join(eval_dir, f'eval_dataset_20_{timestamp}.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(full_20, f)

print(f"\nSaved evaluation dataset ✅")
print(f"  JSON:   {os.path.basename(json_path)}")
print(f"  Pickle: {os.path.basename(pkl_path)}")

# Quick sanity check — both models answered every scenario?
missing = [r['scenario_num'] for r in full_20 
           if not r.get('gemini') or not r.get('nemotron')]
if missing:
    print(f"\n⚠️ Scenarios missing an answer: {missing}")
else:
    print(f"\n✅ All 20 scenarios have both models' answers")

Total scenarios: 20
Scenario numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

Saved evaluation dataset ✅
  JSON:   eval_dataset_20_20260912_150505.json
  Pickle: eval_dataset_20_20260912_150505.pkl

✅ All 20 scenarios have both models' answers
